# Stadium Bowl — Evacuation Path Analysis

Egress distance analysis for `assets/obj/StadiumObj/` using the OBJ + CellComplex + Apertures pipeline.

**Workflow:**
1. Load OBJ solids (stands + concourse) and aperture surfaces (vomitorios + exit)
2. Build `CellComplex.ByFaces` → one cell per stand / concourse volume
3. `Topology.AddApertures` → vomitorios become door-like connections between stands and concourse
4. Build adjacency graph: stand centroid → vomitorio centroid → concourse centroid → exit centroid
5. Shortest path from each stand to nearest exit, worst/avg/best analysis
6. Visualize

## 1. Imports

In [ ]:
import os, time
from topologicpy.Vertex   import Vertex
from topologicpy.Edge     import Edge
from topologicpy.Wire     import Wire
from topologicpy.Face     import Face
from topologicpy.Cell     import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster  import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph    import Graph
import plotly.graph_objects as go

OBJ_DIR  = r"assets/obj/StadiumObj"
RENDERER = "vscode"   # change to 'browser' if not in VS Code
t0 = time.time()

## 2. Load OBJ files

In [ ]:
def load(name):
    return Topology.ByOBJPath(os.path.join(OBJ_DIR, name))

def get_faces(topo_or_list):
    """Flatten a ByOBJPath result (list of topologies) into a flat face list."""
    items = topo_or_list if isinstance(topo_or_list, list) else [topo_or_list]
    faces = []
    for t in items:
        if t is not None:
            faces.extend(Topology.Faces(t))
    return faces

# ── solids ─────────────────────────────────────────────────────────────────────
stand_A_raw      = load("stand_A.obj")
stand_B_raw      = load("stand_B.obj")
stand_C_raw      = load("stand_C.obj")
stand_Corner_raw = load("stand_Corners.obj")
concourse_raw    = load("concourse.obj")

# ── apertures ──────────────────────────────────────────────────────────────────
vomitorio_A_raw  = load("vomitorio_A.obj")
vomitorio_B_raw  = load("vomitorio_B.obj")
vomitorio_C_raw  = load("vomitorio_C.obj")
exit_raw         = load("exit.obj")

print(f"[{time.time()-t0:.1f}s] OBJs loaded")
for label, raw in [
    ("stand_A", stand_A_raw), ("stand_B", stand_B_raw),
    ("stand_C", stand_C_raw), ("stand_Corner", stand_Corner_raw),
    ("concourse", concourse_raw),
    ("vomitorio_A", vomitorio_A_raw), ("vomitorio_B", vomitorio_B_raw),
    ("vomitorio_C", vomitorio_C_raw), ("exit", exit_raw),
]:
    n_faces = len(get_faces(raw))
    print(f"  {label:<15}: {n_faces} faces")

## 3. Build CellComplex

`CellComplex.ByFaces` merges all solid faces into a topological complex — adjacent solids that share a coincident face (at the vomitorio location) get merged into a single complex where those faces become interior shared walls.

Tolerance `0.001` requires exact face coincidence — this is why the Rhino model must have stands and concourse sharing an exact face at each vomitorio opening.

In [ ]:
solid_faces = []
for raw in [stand_A_raw, stand_B_raw, stand_C_raw, stand_Corner_raw, concourse_raw]:
    solid_faces.extend(get_faces(raw))

print(f"Total faces from solids: {len(solid_faces)}")

cellcomplex = CellComplex.ByFaces(solid_faces, tolerance=0.001)

if cellcomplex is None:
    print("ERROR: CellComplex.ByFaces returned None.")
    print("Likely cause: stands and concourse do not share a coincident face.")
    print("Fix: in Rhino, verify each vomitorio opening has an exact shared face between stand and concourse.")
else:
    cells = Topology.Cells(cellcomplex)
    cell_centroids = [Topology.Centroid(c) for c in cells]
    print(f"CellComplex created: {len(cells)} cells")
    for i, (cell, c) in enumerate(zip(cells, cell_centroids)):
        n_faces = len(Topology.Faces(cell))
        print(f"  Cell {i}: {n_faces} faces  centroid=({c.X():.1f}, {c.Y():.1f}, {c.Z():.1f})")

## 4. Add apertures

Vomitorio surfaces and the exit surface are added as apertures to the shared/exterior faces of the CellComplex.  
Tolerance `0.5` allows slight offsets — the aperture face must be coplanar with and contained within the host face.

In [ ]:
vomitorio_faces = get_faces(vomitorio_A_raw) + get_faces(vomitorio_B_raw) + get_faces(vomitorio_C_raw)
exit_faces      = get_faces(exit_raw)
all_apertures   = vomitorio_faces + exit_faces

print(f"Vomitorio faces: {len(vomitorio_faces)}")
print(f"Exit faces:      {len(exit_faces)}")
print(f"Total apertures: {len(all_apertures)}")

cc = Topology.AddApertures(
    cellcomplex,
    all_apertures,
    exclusive=False,
    subTopologyType="face",
    tolerance=0.5
)

# Recompute cells from the aperture-augmented complex
cells = Topology.Cells(cc)
cell_centroids = [Topology.Centroid(c) for c in cells]
print(f"\nApertures added. Cells: {len(cells)}")

# Count how many faces actually received an aperture
faces_with_apertures = 0
for cell in cells:
    for f in Topology.Faces(cell):
        if Topology.Apertures(f):
            faces_with_apertures += 1
print(f"Faces with apertures: {faces_with_apertures}")
if faces_with_apertures == 0:
    print("WARNING: No apertures were assigned. Check vomitorio face coincidence with tolerance=0.5.")

## 5. Label cells

After `CellComplex.ByFaces`, cells are numbered automatically.  
We identify which cell is which stand / concourse by comparing each cell centroid to the centroid of the original solids.

In [ ]:
def first_centroid(raw):
    items = raw if isinstance(raw, list) else [raw]
    xs, ys, zs = [], [], []
    for t in items:
        c = Topology.Centroid(t)
        xs.append(c.X()); ys.append(c.Y()); zs.append(c.Z())
    return (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs))

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** 0.5

reference_centroids = {
    "Stand_A":      first_centroid(stand_A_raw),
    "Stand_B":      first_centroid(stand_B_raw),
    "Stand_C":      first_centroid(stand_C_raw),
    "Stand_Corner": first_centroid(stand_Corner_raw),
    "Concourse":    first_centroid(concourse_raw),
}

cell_labels = []
for c in cell_centroids:
    pt = (c.X(), c.Y(), c.Z())
    best = min(reference_centroids.items(), key=lambda kv: dist3(pt, kv[1]))
    cell_labels.append(best[0])

print("Cell labeling:")
for i, (lbl, c) in enumerate(zip(cell_labels, cell_centroids)):
    print(f"  Cell {i}: {lbl:<15}  ({c.X():.0f}, {c.Y():.0f}, {c.Z():.0f})")

stand_indices    = [i for i, l in enumerate(cell_labels) if l.startswith("Stand")]
concourse_index  = next((i for i, l in enumerate(cell_labels) if l == "Concourse"), None)
print(f"\nStand cells:     {[cell_labels[i] for i in stand_indices]} (indices {stand_indices})")
print(f"Concourse cell:  index {concourse_index}")

## 6. Visualize geometry

3D view: stands colored by type, concourse transparent, vomitorios in blue, exit in green.

In [ ]:
CELL_COLORS = {
    "Stand_A":      (255, 140,  0),
    "Stand_B":      (200,  80,  0),
    "Stand_C":      (255, 190, 50),
    "Stand_Corner": (  0, 170, 110),
    "Concourse":    (200, 210, 230),
}

fig = go.Figure()

# Draw cell face wireframes
for idx, cell in enumerate(cells):
    r, g, b = CELL_COLORS.get(cell_labels[idx], (180, 180, 180))
    opacity = 0.15 if cell_labels[idx] == "Concourse" else 0.35
    for face in Topology.Faces(cell):
        vs = Topology.Vertices(face)
        if len(vs) < 3:
            continue
        xs = [v.X() for v in vs] + [vs[0].X()]
        ys = [v.Y() for v in vs] + [vs[0].Y()]
        zs = [v.Z() for v in vs] + [vs[0].Z()]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(color=f'rgba({r},{g},{b},0.5)', width=1),
            showlegend=False, hoverinfo='skip'
        ))

# Draw cell centroid nodes
for idx, (c, lbl) in enumerate(zip(cell_centroids, cell_labels)):
    r, g, b = CELL_COLORS.get(lbl, (180, 180, 180))
    fig.add_trace(go.Scatter3d(
        x=[c.X()], y=[c.Y()], z=[c.Z()],
        mode='markers+text',
        marker=dict(size=10, color=f'rgb({r},{g},{b})',
                    line=dict(color='white', width=1.5)),
        text=[f'{idx}: {lbl}'], textposition='top center',
        textfont=dict(size=9, color='black'),
        showlegend=False,
        hovertemplate=f'<b>{lbl}</b><br>Cell {idx}<extra></extra>'
    ))

# Draw vomitorio aperture faces (blue)
for f in vomitorio_faces:
    vs = Topology.Vertices(f)
    if len(vs) < 3:
        continue
    xs = [v.X() for v in vs] + [vs[0].X()]
    ys = [v.Y() for v in vs] + [vs[0].Y()]
    zs = [v.Z() for v in vs] + [vs[0].Z()]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='lines',
        line=dict(color='rgba(30,100,200,0.8)', width=3),
        showlegend=False, hoverinfo='skip'
    ))

# Draw exit faces (green)
for f in exit_faces:
    vs = Topology.Vertices(f)
    if len(vs) < 3:
        continue
    xs = [v.X() for v in vs] + [vs[0].X()]
    ys = [v.Y() for v in vs] + [vs[0].Y()]
    zs = [v.Z() for v in vs] + [vs[0].Z()]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='lines',
        line=dict(color='rgba(0,180,0,1.0)', width=4),
        showlegend=False, hoverinfo='skip'
    ))

fig.update_layout(
    title=dict(text='Stadium Bowl — Cell geometry<br><sup>Orange/amber=stands · Gray=concourse · Blue=vomitorios · Green=exit</sup>',
               font=dict(size=14), x=0.5),
    width=1000, height=700,
    scene=dict(
        xaxis=dict(showgrid=False, title='X'),
        yaxis=dict(showgrid=False, title='Y'),
        zaxis=dict(showgrid=False, title='Z'),
        bgcolor='white',
        aspectmode='data'
    ),
    paper_bgcolor='white', showlegend=False
)
fig.show(renderer=RENDERER)

## 7. Build adjacency graph

The graph is built by iterating over all cell pairs, finding shared faces, and routing edges through any apertures on those faces.  
Exit apertures on exterior (non-shared) faces are also collected as destination vertices.

Graph structure:  
`Stand centroid → vomitorio centroid → Concourse centroid → exit centroid`

In [ ]:
graph_verts = list(cell_centroids)
graph_edges = []
vomitorio_waypoints = []   # aperture centroids between cells
exit_waypoints      = []   # aperture centroids on exterior faces (destinations)

# ── cell-to-cell connections via shared apertures ─────────────────────────────
for i in range(len(cells)):
    faces_i = Topology.Faces(cells[i])
    for j in range(i + 1, len(cells)):
        faces_j = Topology.Faces(cells[j])
        for fi in faces_i:
            for fj in faces_j:
                if not Topology.IsSame(fi, fj):
                    continue
                apts = Topology.Apertures(fi)
                if apts:
                    for apt in apts:
                        ac = Topology.Centroid(apt)
                        vomitorio_waypoints.append(ac)
                        graph_edges.append(Edge.ByVertices([cell_centroids[i], ac]))
                        graph_edges.append(Edge.ByVertices([ac, cell_centroids[j]]))
                else:
                    # Shared face with no aperture → direct edge (fallback)
                    graph_edges.append(Edge.ByVertices([cell_centroids[i], cell_centroids[j]]))

# ── exit connections via exterior apertures ───────────────────────────────────
for idx, cell in enumerate(cells):
    for f in Topology.Faces(cell):
        apts = Topology.Apertures(f)
        for apt in apts:
            apt_tops = Topology.ApertureTopologies(apt)
            apt_face = apt_tops[0] if apt_tops else apt
            # Check if this aperture is coplanar with any exit face
            is_exit = any(Face.IsCoplanar(apt_face, ef, tolerance=0.5) for ef in exit_faces)
            if is_exit:
                ac = Topology.Centroid(apt)
                exit_waypoints.append(ac)
                graph_edges.append(Edge.ByVertices([cell_centroids[idx], ac]))

all_verts = graph_verts + vomitorio_waypoints + exit_waypoints
g = Graph.ByVerticesEdges(all_verts, graph_edges)

print(f"Graph: {len(Graph.Vertices(g))} vertices, {len(Graph.Edges(g))} edges")
print(f"  Cell centroids:      {len(cell_centroids)}")
print(f"  Vomitorio waypoints: {len(vomitorio_waypoints)}")
print(f"  Exit waypoints:      {len(exit_waypoints)}")

if len(exit_waypoints) == 0:
    print("\nWARNING: No exit waypoints found.")
    print("The exit.obj surface may not be coplanar with any concourse face (tolerance=0.5).")
    print("Falling back: using concourse centroid as destination.")
    if concourse_index is not None:
        exit_waypoints = [cell_centroids[concourse_index]]

if len(vomitorio_waypoints) == 0:
    print("\nWARNING: No vomitorio connections were found in the graph.")
    print("Check that CellComplex.ByFaces created shared faces at the stand/concourse interface.")

## 8. Shortest paths — stand → nearest exit

In [ ]:
results  = {}
no_path  = []

for idx in stand_indices:
    best = None
    for exit_v in exit_waypoints:
        path = Graph.ShortestPath(g, vertexA=cell_centroids[idx], vertexB=exit_v)
        if path is None:
            continue
        length = Wire.Length(path)
        if best is None or length < best[1]:
            best = (exit_v, length, path)
    if best is None:
        no_path.append(idx)
        print(f"  No path: Cell {idx} ({cell_labels[idx]})")
    else:
        results[idx] = {"exit_v": best[0], "length": best[1], "path": best[2]}
        print(f"  Cell {idx} ({cell_labels[idx]:<15}): {best[1]:.1f} units")

print(f"\nRouted: {len(results)}/{len(stand_indices)} stand cells")

## 9. Worst / average / best case

In [ ]:
lengths = {idx: r["length"] for idx, r in results.items()}

if lengths:
    worst_idx = max(lengths, key=lambda k: lengths[k])
    best_idx  = min(lengths, key=lambda k: lengths[k])
    avg       = sum(lengths.values()) / len(lengths)

    print("=" * 50)
    print(f"Stadium Bowl Egress Analysis — {len(lengths)} stand sections")
    print("=" * 50)
    print(f"  Worst case : {cell_labels[worst_idx]:<15}  {lengths[worst_idx]:8.1f} units")
    print(f"  Best case  : {cell_labels[best_idx]:<15}  {lengths[best_idx]:8.1f} units")
    print(f"  Average    : {'':15}  {avg:8.1f} units")
    print()
    print(f"{'Cell':<5} {'Label':<15} {'Egress distance':>16}")
    print("-" * 38)
    for idx in sorted(lengths, key=lambda k: lengths[k], reverse=True):
        marker = " ← WORST" if idx == worst_idx else (" ← BEST" if idx == best_idx else "")
        print(f"  {idx:<5} {cell_labels[idx]:<15} {lengths[idx]:>12.1f}u{marker}")

if no_path:
    print(f"\nStand cells with NO path to exit: {[cell_labels[i] for i in no_path]}")
    print("Check vomitorio aperture coincidence and graph connectivity.")

## 10. Graph visualization — 3D nodes + edges + paths

In [ ]:
def len_to_color(length, mn, mx):
    t = 0.0 if mx <= mn else max(0.0, min(1.0, (length - mn) / (mx - mn)))
    if t < 0.5:
        return (int(510 * t), 200, 0)
    return (255, int(200 * (1 - (t - 0.5) * 2)), 0)

fig2 = go.Figure()

mn = min(lengths.values()) if lengths else 0
mx = max(lengths.values()) if lengths else 1

# ── all graph edges (thin gray) ────────────────────────────────────────────────
for e in Graph.Edges(g):
    vs = Topology.Vertices(e)
    if len(vs) == 2:
        fig2.add_trace(go.Scatter3d(
            x=[vs[0].X(), vs[1].X()],
            y=[vs[0].Y(), vs[1].Y()],
            z=[vs[0].Z(), vs[1].Z()],
            mode='lines',
            line=dict(color='rgba(120,120,140,0.3)', width=1.5),
            showlegend=False, hoverinfo='skip'
        ))

# ── all graph nodes (white circles) ───────────────────────────────────────────
for v in Graph.Vertices(g):
    fig2.add_trace(go.Scatter3d(
        x=[v.X()], y=[v.Y()], z=[v.Z()],
        mode='markers',
        marker=dict(size=5, color='white',
                    line=dict(color='rgba(80,80,100,0.7)', width=1.5)),
        showlegend=False, hoverinfo='skip'
    ))

# ── egress paths (colored by distance) ────────────────────────────────────────
for idx, r in results.items():
    path_verts = Topology.Vertices(r["path"])
    if len(path_verts) < 2:
        continue
    col = len_to_color(r["length"], mn, mx)
    is_worst = (idx == worst_idx)
    fig2.add_trace(go.Scatter3d(
        x=[v.X() for v in path_verts],
        y=[v.Y() for v in path_verts],
        z=[v.Z() for v in path_verts],
        mode='lines+markers',
        line=dict(color=f'rgb{col}', width=7 if is_worst else 3),
        marker=dict(
            size=10 if is_worst else 7,
            color='white',
            line=dict(color=f'rgb{col}', width=2.5 if is_worst else 1.8)
        ),
        showlegend=False,
        hovertemplate=(
            f"<b>{cell_labels[idx]}</b><br>"
            f"Egress: {r['length']:.1f} units"
            "<extra></extra>"
        )
    ))

# ── cell centroid labels (large markers for stands) ────────────────────────────
for idx, (c, lbl) in enumerate(zip(cell_centroids, cell_labels)):
    r_c, g_c, b_c = CELL_COLORS.get(lbl, (180, 180, 180))
    fig2.add_trace(go.Scatter3d(
        x=[c.X()], y=[c.Y()], z=[c.Z()],
        mode='markers+text',
        marker=dict(size=14 if lbl.startswith('Stand') else 10,
                    color=f'rgb({r_c},{g_c},{b_c})',
                    line=dict(color='white', width=2)),
        text=[lbl], textposition='top center',
        textfont=dict(size=9, color='black'),
        showlegend=False,
        hovertemplate=f'<b>{lbl}</b><br>Cell {idx}<extra></extra>'
    ))

# ── exit waypoints (green stars) ──────────────────────────────────────────────
for ev in exit_waypoints:
    fig2.add_trace(go.Scatter3d(
        x=[ev.X()], y=[ev.Y()], z=[ev.Z()],
        mode='markers+text',
        marker=dict(size=15, color='green', symbol='diamond',
                    line=dict(color='darkgreen', width=2)),
        text=['EXIT'], textposition='top center',
        textfont=dict(size=10, color='darkgreen'),
        showlegend=False,
        hovertemplate='EXIT<extra></extra>'
    ))

# ── worst-case star ────────────────────────────────────────────────────────────
if lengths:
    wc = cell_centroids[worst_idx]
    fig2.add_trace(go.Scatter3d(
        x=[wc.X()], y=[wc.Y()], z=[wc.Z()],
        mode='markers+text',
        marker=dict(size=18, color='darkred', symbol='diamond'),
        text=[f'WORST: {cell_labels[worst_idx]} ({lengths[worst_idx]:.0f}u)'],
        textposition='bottom center',
        textfont=dict(size=10, color='darkred'),
        showlegend=False
    ))

# ── colorbar legend ────────────────────────────────────────────────────────────
fig2.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None], mode='markers',
    marker=dict(
        colorscale=[[0,'rgb(0,200,0)'],[0.5,'rgb(255,200,0)'],[1,'rgb(255,0,0)']],
        cmin=round(mn, 1), cmax=round(mx, 1), color=[mn, mx],
        colorbar=dict(title=dict(text='Egress (units)', side='right'),
                      thickness=14, len=0.5, x=1.0),
        showscale=True
    ), showlegend=False
))

fig2.update_layout(
    title=dict(
        text='Stadium Bowl — Egress graph + paths<br>'
             '<sup>Green=short path · Red=long path · Diamond=exit · Colored nodes=stand/concourse</sup>',
        font=dict(size=14), x=0.5
    ),
    width=1100, height=750,
    scene=dict(
        xaxis=dict(showgrid=False, title='X'),
        yaxis=dict(showgrid=False, title='Y'),
        zaxis=dict(showgrid=False, title='Z'),
        bgcolor='white',
        aspectmode='data'
    ),
    paper_bgcolor='white', showlegend=False
)
fig2.show(renderer=RENDERER)

## 11. Limitations

- **Units**: distances are in Rhino model units (check your Rhino document units — likely centimetres).
- **Centroid-to-centroid routing**: paths go stand centroid → vomitorio centroid → concourse centroid → exit. They do not trace the actual walked path across stepped seating rows.
- **One cell per stand section**: the entire seating volume of each stand is treated as one node. For finer analysis, the stand would need to be subdivided into row-level cells.
- **No congestion**: uniform walking speed, no queuing at vomitorios.